# Advanced PyVis Features

This notebook covers power-user features: context managers, the iterator protocol, node groups, and CDN resource modes.

**What you'll learn:**
- Using `Network` as a context manager
- Iterating over networks (len, in, for)
- Node groups for categorical coloring
- CDN resource modes (in_line, remote, local)
- Setting custom vis.js options via JSON

In [ ]:
from pyvis.network import Network

## 1. Context Manager

`Network` supports Python's `with` statement. This is convenient for building and showing a network in a single block.

In [ ]:
with Network(notebook=True, cdn_resources="in_line", height="400px") as net:
    net.add_node(1, label="Alpha", color="#6366f1")
    net.add_node(2, label="Beta", color="#22d3ee")
    net.add_node(3, label="Gamma", color="#34d399")
    net.add_node(4, label="Delta", color="#f87171")
    net.add_edge(1, 2, width=2)
    net.add_edge(2, 3, width=2)
    net.add_edge(3, 4, width=2)
    net.add_edge(4, 1, width=2)
    net.add_edge(1, 3, width=1)
    net.show("04_context.html")

## 2. Iterator Protocol

`Network` supports `len()`, the `in` operator, and `for` loops over node IDs. This makes it easy to query and inspect a network.

In [ ]:
net = Network(notebook=True, cdn_resources="in_line")
for i in range(1, 11):
    net.add_node(i, label=f"N{i}")
for i in range(1, 10):
    net.add_edge(i, i + 1)
net.add_edge(10, 1)

print(f"Number of nodes: {len(net)}")
print(f"Node 5 exists: {5 in net}")
print(f"Node 99 exists: {99 in net}")
print(f"All node IDs: {list(net)}")

## 3. Node Groups

Assign nodes to groups for automatic categorical coloring. vis.js assigns a distinct color to each group.

In [ ]:
net = Network(notebook=True, cdn_resources="in_line", height="500px")

# Infrastructure groups
infra = {
    "server": ["Web Server 1", "Web Server 2", "API Gateway"],
    "database": ["PostgreSQL", "Redis", "Elasticsearch"],
    "client": ["Browser", "Mobile App", "CLI"],
}

node_id = 1
ids = {}
for group, members in infra.items():
    for member in members:
        net.add_node(node_id, label=member, group=group, title=f"{member} ({group})")
        ids[member] = node_id
        node_id += 1

# Connections
connections = [
    ("Browser", "Web Server 1"), ("Mobile App", "Web Server 2"), ("CLI", "API Gateway"),
    ("Web Server 1", "API Gateway"), ("Web Server 2", "API Gateway"),
    ("API Gateway", "PostgreSQL"), ("API Gateway", "Redis"), ("API Gateway", "Elasticsearch"),
]
for src, dst in connections:
    net.add_edge(ids[src], ids[dst])

net.show("04_groups.html")

## 4. CDN Resource Modes

PyVis supports three modes for including the vis.js library:

| Mode | `cdn_resources=` | File size | Requires internet |
|------|-------------------|-----------|-------------------|
| Inline | `"in_line"` | ~1.5 MB | No |
| Remote | `"remote"` | ~5 KB | Yes |
| Local | `"local"` | ~5 KB | No (needs vis.js files alongside) |

For Jupyter notebooks, use `"in_line"` or `"remote"`. The `"local"` mode is best for standalone HTML files deployed alongside the vis.js library.

In [ ]:
# Remote CDN — small file, loads vis.js from unpkg.com
net = Network(notebook=True, cdn_resources="remote", height="300px")
net.add_node(1, label="Remote CDN", color="#22d3ee", size=30)
net.add_node(2, label="Fast load", color="#34d399", size=20)
net.add_edge(1, 2)
net.show("04_remote.html")

## 5. Custom vis.js JSON Options

For options not yet covered by `pyvis.types`, you can pass raw JSON strings to `set_options()`. This accepts the same format as the vis.js configuration UI.

In [ ]:
import networkx as nx

G = nx.barabasi_albert_graph(20, 2, seed=42)
net = Network(notebook=True, cdn_resources="in_line", height="400px")
net.from_nx(G)

net.set_options('''
var options = {
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -5000,
      "springLength": 200,
      "springConstant": 0.001
    },
    "maxVelocity": 30,
    "minVelocity": 0.75
  },
  "edges": {
    "color": {
      "inherit": "both"
    },
    "smooth": {
      "type": "continuous"
    }
  }
}
''')
net.show("04_json_options.html")

## 6. Directed Graphs

Set `directed=True` to add arrows to edges.

In [ ]:
net = Network(notebook=True, cdn_resources="in_line", height="400px", directed=True)

# A dependency graph
deps = {
    "App": ["Router", "Store"],
    "Router": ["Auth", "API"],
    "Store": ["API"],
    "Auth": ["API"],
    "API": ["Database"],
}

colors = {"App": "#f87171", "Router": "#fbbf24", "Store": "#fbbf24",
          "Auth": "#34d399", "API": "#22d3ee", "Database": "#818cf8"}
for node, targets in deps.items():
    net.add_node(node, label=node, color=colors.get(node, "#6366f1"), size=25, shape="box")
    for target in targets:
        net.add_node(target, label=target, color=colors.get(target, "#6366f1"), size=25, shape="box")
        net.add_edge(node, target)

net.show("04_directed.html")

---
**Previous:** [03_typed_options.ipynb](03_typed_options.ipynb) | **Start:** [01_basics.ipynb](01_basics.ipynb)